# 📝 GIẢI BÀI TẬP - Vector Embeddings và Độ Tương Đồng Ngữ Nghĩa

File này chứa lời giải mẫu cho các bài tập trong `embedding_ex.ipynb`.

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install sentence-transformers numpy pandas -q

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

# Load model đa ngôn ngữ (dùng chung cho tất cả bài tập)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Model loaded!")

# ---- 3 hàm đo độ tương đồng cơ bản ----

def cosine_similarity(vec1, vec2):
    """Cosine similarity: đo góc giữa 2 vectors. Range [-1, 1], càng gần 1 càng giống nhau."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def euclidean_distance(vec1, vec2):
    """Euclidean distance: đo khoảng cách giữa 2 vectors. Range [0, ∞], càng gần 0 càng giống nhau."""
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

def dot_product(vec1, vec2):
    """Dot product: tích vô hướng. Càng lớn càng giống nhau."""
    return np.dot(vec1, vec2)

print("✅ Các hàm đo độ tương đồng đã sẵn sàng!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!
✅ Các hàm đo độ tương đồng đã sẵn sàng!


---
## Bài tập 1: Tạo embeddings và tính độ tương đồng

In [ ]:
# Bài tập 1: 3 câu về chủ đề AI
my_sentences = [
    "Machine learning giúp máy tính học từ dữ liệu",
    "Deep learning là một nhánh của machine learning",
    "Hôm nay trời nắng đẹp"
]

# Bước 1: Tạo embeddings
my_embeddings = model.encode(my_sentences)

# Bước 2: Tính độ tương đồng giữa các cặp câu
vec1 = my_embeddings[0]
vec2 = my_embeddings[1]
vec3 = my_embeddings[2]

# Cosine similarity giữa câu 1 và 2
cos_12 = cosine_similarity(vec1, vec2)
print(f"Cosine similarity (câu 1 vs câu 2): {cos_12:.4f}")

# Euclidean distance giữa câu 1 và 3
euc_13 = euclidean_distance(vec1, vec3)
print(f"Euclidean distance (câu 1 vs câu 3): {euc_13:.4f}")

# Dot product giữa câu 2 và 3
dot_23 = dot_product(vec2, vec3)
print(f"Dot product       (câu 2 vs câu 3): {dot_23:.4f}")

# Bước 3: Phân tích kết quả
print("\n--- Phân tích ---")
print(f"Câu 1 và câu 2 cùng chủ đề AI  {cos_12:.4f}")
print(f"Câu 1 và câu 3 khác chủ đề   {euc_13:.4f}")
print(f"Câu 2 và câu 3 khác chủ đề   {dot_23:.4f}")

Số câu: 3
Kích thước mỗi vector: 384 chiều

Cosine similarity (câu 1 vs câu 2): 0.7565
Euclidean distance (câu 1 vs câu 3): 6.8704
Dot product       (câu 2 vs câu 3): -2.1441

--- Phân tích ---
Câu 1 và câu 2 cùng chủ đề AI → cosine similarity cao:  0.7565
Câu 1 và câu 3 khác chủ đề    → euclidean distance lớn: 6.8704
Câu 2 và câu 3 khác chủ đề    → dot product thấp hơn:   -2.1441


---
## Bài tập 2: Hàm so sánh 3 phương pháp đo độ tương đồng

In [3]:
def compare_similarity_methods(sentence1, sentence2, model):
    """
    So sánh 3 phương pháp đo độ tương đồng giữa 2 câu.

    Returns:
        DataFrame với kết quả của cả 3 phương pháp
    """
    # Tạo embeddings cho 2 câu
    embeddings = model.encode([sentence1, sentence2])
    vec1 = embeddings[0]
    vec2 = embeddings[1]

    # Tính toán với 3 phương pháp
    cos_score  = cosine_similarity(vec1, vec2)
    euc_score  = euclidean_distance(vec1, vec2)
    dot_score  = dot_product(vec1, vec2)

    # Tạo DataFrame và return
    results = {
        'Phương pháp': ['Cosine Similarity', 'Euclidean Distance', 'Dot Product'],
        'Score':       [round(cos_score, 4),  round(euc_score, 4),  round(dot_score, 4)],
        'Ý nghĩa':     ['Càng gần 1 càng giống', 'Càng gần 0 càng giống', 'Càng lớn càng giống'],
    }
    return pd.DataFrame(results)


# Test hàm
sentence1 = "Học máy là một lĩnh vực của AI"
sentence2 = "Machine learning is a field of artificial intelligence"

df_result = compare_similarity_methods(sentence1, sentence2, model)
print(f"Câu 1: {sentence1}")
print(f"Câu 2: {sentence2}\n")
print(df_result.to_string(index=False))

Câu 1: Học máy là một lĩnh vực của AI
Câu 2: Machine learning is a field of artificial intelligence

       Phương pháp   Score               Ý nghĩa
 Cosine Similarity  0.7292 Càng gần 1 càng giống
Euclidean Distance  3.3974 Càng gần 0 càng giống
       Dot Product 15.5356   Càng lớn càng giống


---
## Bài tập 3: Hệ thống tìm kiếm câu hỏi-đáp

In [4]:
# Knowledge base về chủ đề AI / Machine Learning (10 câu)
my_knowledge_base = [
    "Python là ngôn ngữ lập trình phổ biến nhất cho AI và Machine Learning",
    "Machine Learning là lĩnh vực giúp máy tính học từ dữ liệu mà không cần lập trình cứng",
    "Deep Learning sử dụng mạng nơ-ron nhiều lớp để học các đặc trưng phức tạp",
    "TensorFlow và PyTorch là 2 framework deep learning phổ biến nhất hiện nay",
    "GPT là mô hình ngôn ngữ lớn do OpenAI phát triển",
    "Vector embedding biểu diễn ý nghĩa của từ hoặc câu dưới dạng mảng số",
    "RAG (Retrieval-Augmented Generation) kết hợp tìm kiếm tài liệu với mô hình ngôn ngữ",
    "ChromaDB là vector database dùng để lưu trữ và tìm kiếm embedding",
    "Fine-tuning là quá trình huấn luyện lại một phần mô hình đã được huấn luyện trước",
    "Cosine similarity là phép đo phổ biến nhất để so sánh độ tương đồng giữa 2 vectors",
]

# Bước 1: Tạo embeddings cho toàn bộ knowledge base
kb_embeddings = model.encode(my_knowledge_base)
print(f"✅ Đã tạo embeddings cho {len(my_knowledge_base)} câu trong knowledge base\n")

# Bước 2: Hàm tìm kiếm top-k câu liên quan nhất
def search(query, knowledge_base, kb_embeddings, top_k=3):
    # Tạo embedding cho câu hỏi
    query_embedding = model.encode([query])[0]

    # Tính cosine similarity giữa query và từng câu trong knowledge base
    scores = []
    for i, kb_emb in enumerate(kb_embeddings):
        score = cosine_similarity(query_embedding, kb_emb)
        scores.append((score, i))

    # Sắp xếp theo score giảm dần, lấy top_k
    scores.sort(reverse=True)

    # In kết quả
    print(f"❓ Query: {query}\n")
    for rank, (score, idx) in enumerate(scores[:top_k], start=1):
        print(f"  Top {rank} (score={score:.4f}): {knowledge_base[idx]}")

# Bước 3: Thử tìm kiếm với câu hỏi của người dùng
user_query = "Làm thế nào để máy tính học từ dữ liệu?"
search(user_query, my_knowledge_base, kb_embeddings, top_k=3)

✅ Đã tạo embeddings cho 10 câu trong knowledge base

❓ Query: Làm thế nào để máy tính học từ dữ liệu?

  Top 1 (score=0.5107): Machine Learning là lĩnh vực giúp máy tính học từ dữ liệu mà không cần lập trình cứng
  Top 2 (score=0.3841): RAG (Retrieval-Augmented Generation) kết hợp tìm kiếm tài liệu với mô hình ngôn ngữ
  Top 3 (score=0.3408): ChromaDB là vector database dùng để lưu trữ và tìm kiếm embedding


---
## Bài tập 4: So sánh các Embedding Models

In [5]:
import time

# Load 3 models để so sánh
models_to_compare = {
    "multilingual-MiniLM (384d)": "paraphrase-multilingual-MiniLM-L12-v2",
    "MiniLM-L6 English (384d)":   "all-MiniLM-L6-v2",
    "multilingual-mpnet (768d)":  "paraphrase-multilingual-mpnet-base-v2",
}

# 2 câu để so sánh (cùng nghĩa, khác ngôn ngữ)
sentence_1 = "Artificial intelligence is the future"
sentence_2 = "AI sẽ định hình tương lai của chúng ta"

print(f"Câu 1: {sentence_1}")
print(f"Câu 2: {sentence_2}\n")
print(f"{'Model':<35} {'Dim':>5}  {'Cosine':>8}  {'Thời gian':>10}")
print("-" * 65)

for name, model_id in models_to_compare.items():
    # Load model và đo thời gian encode
    m = SentenceTransformer(model_id)

    start = time.time()
    vecs = m.encode([sentence_1, sentence_2])
    elapsed = time.time() - start

    score = cosine_similarity(vecs[0], vecs[1])
    dim   = vecs.shape[1]

    print(f"{name:<35} {dim:>5}  {score:>8.4f}  {elapsed*1000:>8.1f} ms")


Câu 1: Artificial intelligence is the future
Câu 2: AI sẽ định hình tương lai của chúng ta

Model                                 Dim    Cosine   Thời gian
-----------------------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


multilingual-MiniLM (384d)            384    0.7749      31.0 ms


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

multilingual-mpnet (768d)             768    0.8209     288.1 ms
